# Q-MetaIDS: Colab-ready Notebook

**Quantum-Inspired Meta-Learning for Privacy-Preserving, Energy-Aware Few-Shot Cross-Domain IoT Intrusion Detection**

This notebook provides a runnable, Colab-friendly PyTorch implementation scaffold for:

- Dataset preprocessing & virtual client simulation (dataset-only experiments)
- Quantum-inspired feature map (classical simulation)
- Quantum-Inspired Meta-Optimizer (QIMO)
- Federated meta-training loop (simulated clients)
- Differential Privacy (clip + Gaussian noise) for client updates
- Analytic energy estimator (FLOPs & bytes)

**How to use**:
1. Run each cell sequentially in Google Colab. 
2. Upload dataset files to the `/content/data/` folder or change the dataset loading cell to download datasets directly.
3. Tune hyperparameters at the **CONFIG** cell.

---

*This notebook is a complete scaffold — replace dataset loading cells with your dataset paths (TON-IoT, UNSW-NB15, CIC-IDS2017, etc.) and run the end-to-end experiments.*

In [ ]:
# Install required packages (run in Colab)
!pip install torch torchvision --quiet
!pip install numpy pandas scikit-learn matplotlib tqdm --quiet
!pip install opacus --quiet
# Flower is optional for advanced federated simulation
!pip install flwr --quiet

print('Packages installed. Restart the runtime if necessary (only if major C-extension errors occur).')

In [ ]:
# Imports and basic config
import os
import math
import random
from copy import deepcopy
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

# Create data directory
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)


In [ ]:
# CONFIG: change these for your experiments
CONFIG = {
    'num_virtual_clients': 8,
    'meta_rounds': 20,
    'clients_per_round': 4,
    'inner_steps': 3,
    'inner_lr': 0.01,
    'meta_lr': 0.001,
    'qimo_beta': 0.1,
    'qimo_phi': 0.5,
    'clip_norm': 1.0,
    'dp_noise_multiplier': 0.5,  # sigma
    'top_k': 0.3,  # fraction of weights to send (sparsification)
    'quantize_bits': 8,
    'batch_size': 32,
    'k_shot': 5,
}

print('CONFIG set. Update as needed.')

In [ ]:
# Example: synthetic dataset generator (quick test). Replace with real dataset loading.

def generate_synthetic_dataset(n_samples=5000, n_features=20, n_classes=2):
    X = np.random.randn(n_samples, n_features)
    # Create an imbalanced scenario with rare attack class
    y = (np.sum(X[:, :3], axis=1) + 0.5*np.random.randn(n_samples) > 0).astype(int)
    return X.astype(np.float32), y.astype(np.int64)

X, y = generate_synthetic_dataset(n_samples=4000, n_features=30, n_classes=2)
print('Synthetic data shapes:', X.shape, y.shape)

# Partition into virtual clients
def partition_clients(X, y, num_clients=CONFIG['num_virtual_clients']):
    data_per_client = int(len(X) / num_clients)
    clients = {}
    idxs = np.arange(len(X))
    np.random.shuffle(idxs)
    for i in range(num_clients):
        s = i*data_per_client
        e = s + data_per_client if i < num_clients-1 else len(X)
        sel = idxs[s:e]
        clients[i] = {'X': X[sel], 'y': y[sel]}
    return clients

clients = partition_clients(X, y)
print('Created', len(clients), 'virtual clients.')

In [ ]:
# Preprocessing helpers (scaling, tensor conversion)
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def get_data_loaders_for_client(client_data, batch_size=32, test_split=0.2):
    X = client_data['X']
    y = client_data['y']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_split, random_state=42, stratify=y)
    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr)
    X_te = scaler.transform(X_te)
    train_loader = DataLoader(SimpleDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(SimpleDataset(X_te, y_te), batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, scaler

# Example for client 0
train_loader0, test_loader0, scaler0 = get_data_loaders_for_client(clients[0], batch_size=CONFIG['batch_size'])
print('Client 0 train batches:', len(train_loader0))

In [ ]:
# Model: lightweight encoder + classifier (suitable for IoT)
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.fc2(x))
        return x

class Classifier(nn.Module):
    def __init__(self, feat_dim, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        return self.fc(x)

# Combined model wrapper
class QMetaModel(nn.Module):
    def __init__(self, input_dim, feat_dim=128, num_classes=2):
        super().__init__()
        self.encoder = Encoder(input_dim, feat_dim)
        self.classifier = Classifier(feat_dim, num_classes)
    def forward(self, x):
        z = self.encoder(x)
        out = self.classifier(z)
        return out

# Quick instantiation
model = QMetaModel(input_dim=clients[0]['X'].shape[1]).to(DEVICE)
print(model)

In [ ]:
# QIMO inner update: perform S gradient steps with quantum-inspired rotation

def qimo_adapt(model, loss_fn, support_loader, inner_steps, inner_lr, beta=0.1, phi=0.5, device='cpu'):
    # Create a copy of model parameters (fast weights)
    fast_model = deepcopy(model)
    fast_model.to(device)
    for step in range(inner_steps):
        fast_model.train()
        for xb, yb in support_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = fast_model(xb)
            loss = loss_fn(preds, yb)
            # compute grads
            grads = torch.autograd.grad(loss, fast_model.parameters(), create_graph=False)
            # apply gradient descent manually plus rotation term
            with torch.no_grad():
                # gather grads into a vector and apply update per-parameter
                for p, g in zip(fast_model.parameters(), grads):
                    if g is None:
                        continue
                    # gradient step
                    p -= inner_lr * g
                    # quantum-inspired rotation: small orthogonal perturbation
                    g_norm = g.norm() if g.norm().item() != 0 else torch.tensor(1.0, device=g.device)
                    g_hat = g / g_norm
                    # create small orthogonal perturbation u (random noise orthogonalized)
                    u = torch.randn_like(g)
                    u = u - (torch.dot(u.view(-1), g_hat.view(-1)))*g_hat.view(-1)
                    if u.norm().item() == 0:
                        u = torch.randn_like(g)
                    u_hat = u / (u.norm() + 1e-12)
                    rot = beta * (math.cos(phi)*g_hat + math.sin(phi)*u_hat)
                    p -= rot
    return fast_model

# Test qimo_adapt with client 0 small support set
support_loader = DataLoader(SimpleDataset(clients[0]['X'][:CONFIG['k_shot']], clients[0]['y'][:CONFIG['k_shot']]), batch_size=CONFIG['k_shot'])
loss_fn = nn.CrossEntropyLoss()
fast = qimo_adapt(model, loss_fn, support_loader, inner_steps=CONFIG['inner_steps'], inner_lr=CONFIG['inner_lr'], beta=CONFIG['qimo_beta'], phi=CONFIG['qimo_phi'], device=DEVICE)
print('Adaptation completed (fast model ready).')

In [ ]:
# Utilities: compress, clip, add Gaussian DP noise
import copy

def get_update_and_size(model_old, model_new):
    # returns vectorized update and its byte size approximation
    vec = []
    for p_old, p_new in zip(model_old.parameters(), model_new.parameters()):
        delta = (p_new.data - p_old.data).view(-1).cpu().numpy()
        vec.append(delta)
    vec = np.concatenate(vec)
    bytes_est = vec.nbytes
    return vec, bytes_est

def top_k_sparsify(vec, fraction=0.3):
    k = int(len(vec) * fraction)
    if k < 1:
        return vec
    idx = np.abs(vec).argsort()[-k:]
    sparse = np.zeros_like(vec)
    sparse[idx] = vec[idx]
    return sparse

def clip_and_add_noise(vec, clip_norm=1.0, noise_multiplier=0.5):
    norm = np.linalg.norm(vec)
    if norm > clip_norm:
        vec = vec * (clip_norm / norm)
    noise = np.random.normal(loc=0.0, scale=noise_multiplier*clip_norm, size=vec.shape)
    return vec + noise

# Test utilities
vec, b = get_update_and_size(model, fast)
print('Update vector length:', len(vec), 'Bytes estimate:', b)
vec_s = top_k_sparsify(vec, CONFIG['top_k'])
vec_noisy = clip_and_add_noise(vec_s, CONFIG['clip_norm'], CONFIG['dp_noise_multiplier'])
print('Sparsified and DP-noised update ready.')

In [ ]:
# Energy estimator (analytic)
# User can adjust eta_op and eta_byte per literature
ETA_OP = 1e-9  # J per FLOP (example proxy)
ETA_BYTE = 1e-7  # J per byte transferred

# Rough FLOPs estimate for a forward+backward pass for a model (very approximate)
def estimate_flops(model, input_dim, batch_size=1):
    # crude estimate: sum 2*in*out for linear layers
    flops = 0
    for p in model.parameters():
        if p.ndim >= 2:
            flops += 2 * np.prod(p.shape)
    # scale by batch
    return flops * batch_size

# example
flops_example = estimate_flops(model, clients[0]['X'].shape[1], batch_size=CONFIG['batch_size'])
print('Estimated FLOPs per pass (approx):', flops_example)
E_comp = ETA_OP * flops_example
print('Estimated energy per pass (J):', E_comp)

In [ ]:
# Federated meta-training skeleton (dataset-only simulation)

def vector_to_state_dict(vec, model_template):
    # reconstruct state_dict-like list of tensors from flat vec
    state = {}
    pointer = 0
    for name, p in model_template.named_parameters():
        numel = p.numel()
        arr = vec[pointer:pointer+numel]
        tensor = torch.tensor(arr.reshape(p.shape), dtype=p.dtype)
        state[name] = tensor
        pointer += numel
    return state

def apply_sparse_update_to_model(model, vec):
    # apply flattened numpy vector update to model parameters (in-place add)
    pointer = 0
    for p in model.parameters():
        numel = p.numel()
        delta = vec[pointer:pointer+numel].reshape(p.shape)
        p.data += torch.tensor(delta, dtype=p.data.dtype, device=p.data.device)
        pointer += numel

def federated_meta_train(clients_data, global_model, config):
    meta_optimizer = torch.optim.Adam(global_model.parameters(), lr=config['meta_lr'])
    loss_fn = nn.CrossEntropyLoss()
    for r in range(config['meta_rounds']):
        selected = random.sample(list(clients_data.keys()), config['clients_per_round'])
        updates = []
        sizes = []
        energies = []
        for cid in selected:
            client = clients_data[cid]
            # prepare small support loader for few-shot
            Xs = client['X'][:config['k_shot']]
            ys = client['y'][:config['k_shot']]
            support_loader = DataLoader(SimpleDataset(scaler0.transform(Xs), ys), batch_size=len(Xs))
            # local adaptation via QIMO
            fast_model = qimo_adapt(global_model, loss_fn, support_loader, config['inner_steps'], config['inner_lr'], beta=config['qimo_beta'], phi=config['qimo_phi'], device=DEVICE)
            # compute update
            vec, bts = get_update_and_size(global_model, fast_model)
            # sparsify
            vec_sp = top_k_sparsify(vec, config['top_k'])
            # clip and add DP noise
            vec_dp = clip_and_add_noise(vec_sp, config['clip_norm'], config['dp_noise_multiplier'])
            updates.append(vec_dp)
            sizes.append(bts)
            # estimate energy (comp + comm)
            flops = estimate_flops(global_model, client['X'].shape[1], batch_size=len(Xs)) * config['inner_steps']
            E = ETA_OP * flops + ETA_BYTE * bts
            energies.append(E)
        # Aggregate updates (weighted by energy factor inverse to prefer low energy)
        weights = np.exp(-np.array(energies))
        weights = weights / weights.sum()
        agg = np.zeros_like(updates[0])
        for w, u in zip(weights, updates):
            agg += w * u
        # apply aggregated update to global model
        apply_sparse_update_to_model(global_model, agg)
        # optional meta-optimizer step (here we already updated params directly)
        print(f'Round {r+1}/{config["meta_rounds"]}: applied agg update. avg energy:', np.mean(energies))
    return global_model

# Run a short federated meta-train on synthetic clients to validate
print('Starting quick federated meta-training test (few rounds)...')
global_model = QMetaModel(input_dim=clients[0]['X'].shape[1]).to(DEVICE)
trained = federated_meta_train(clients, global_model, {**CONFIG, 'meta_rounds':3, 'clients_per_round':3})
print('Federated meta-training test completed.')

In [ ]:
# Evaluation: k-shot adaptation and test function

def evaluate_k_shot(global_model, target_client_data, k_list=[1,5,10], inner_steps=3):
    results = {}
    loss_fn = nn.CrossEntropyLoss()
    for k in k_list:
        # prepare support and query
        X = target_client_data['X']
        y = target_client_data['y']
        if len(X) < k+10:
            results[k] = {'acc': None}
            continue
        idx = np.random.choice(len(X), size=k+50, replace=False)
        sup_idx = idx[:k]
        qry_idx = idx[k:]
        support_loader = DataLoader(SimpleDataset(scaler0.transform(X[sup_idx]), y[sup_idx]), batch_size=k)
        query_loader = DataLoader(SimpleDataset(scaler0.transform(X[qry_idx]), y[qry_idx]), batch_size=32)
        adapted = qimo_adapt(global_model, loss_fn, support_loader, inner_steps, CONFIG['inner_lr'], CONFIG['qimo_beta'], CONFIG['qimo_phi'], DEVICE)
        # test
        adapted.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in query_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = adapted(xb)
                _, predicted = torch.max(preds, 1)
                correct += (predicted == yb).sum().item()
                total += yb.size(0)
        acc = correct/total if total>0 else None
        results[k] = {'acc': acc}
    return results

# Quick eval on client 1
res = evaluate_k_shot(trained, clients[1], k_list=[1,5,10], inner_steps=CONFIG['inner_steps'])
print('k-shot eval results (synthetic):', res)